# Step 3.6: reproduce the QBR numbers from the clean data

The Finance QBR (`06_qbr_finance_and_mis_pack.pdf`, p1 scorecard, p2 sections 2 and 3.1) quotes five quarters of figures.
If our clean data gives the **same** numbers, we trust both the data and our reading of the columns. If it does not, the gap is a finding.

**Run `step3_clean.ipynb` first.** This notebook reads `data-clean/`, changes nothing there, and writes
`data-clean/audit/qbr_reconciliation.csv` and `notes/qbr_reconciliation.md`.

Quarters are financial-year quarters: Q1 = Apr to Jun (`06` p1). FY27 Q1 = Apr to Jun 2026.

In [11]:
import os
import numpy as np, pandas as pd
CLEAN = os.path.join('..', 'data-clean'); AUD = os.path.join(CLEAN, 'audit'); NOTES = os.path.join('..', 'notes')
if not os.path.isdir(AUD): raise FileNotFoundError('Run step3_clean.ipynb first.')
rd = lambda n, d=[]: pd.read_csv(os.path.join(CLEAN, n + '.csv'), keep_default_na=False, na_values=[''], parse_dates=d)
ps = rd('primary_sales', ['invoice_date']); ec = rd('expiry_claims', ['claim_date', 'credit_note_date'])
sn = rd('cnf_stock_snapshot', ['snapshot_date']); sk = rd('sku_master')
PTS = sk.set_index('sku_code').pts_inr

def fq(s):                      # financial-year quarter label, e.g. 2026-05-10 -> FY27Q1
    d = s.dt; fy = d.year.where(d.month < 4, d.year + 1); q = ((d.month - 4) % 12) // 3 + 1
    return 'FY' + (fy % 100).astype(str) + 'Q' + q.astype(str)

QS = ['FY26Q1', 'FY26Q2', 'FY26Q3', 'FY26Q4', 'FY27Q1']
# What the QBR says (06 p1 scorecard). Values are exactly as printed.
QBR = {
 'Primary sales, gross at PTS (INR crore)': [52.52, 51.84, 50.42, 49.37, 51.36],
 'FOC under QS schemes, % of primary':       [4.68, 3.75, 3.47, 3.19, 3.69],
 'NE/expiry claims logged (INR crore)':      [0.62, 0.85, 1.21, 1.15, 1.41],
 'Claims as % of primary sales':             [1.18, 1.64, 2.41, 2.33, 2.75],
 'Fill rate to stockists (qty), %':          [99.5, 94.6, 91.5, 90.3, 90.9],
 'Claim-to-CN TAT, days':                    [47, 46, 44, 45, 46],
 'CFA inventory, quarter end at PTS (INR crore)': [17.49, 15.35, 13.89, 13.14, 14.05],
}
print(ps.shape, ec.shape, sn.shape)

(93063, 16) (4503, 21) (5015, 11)


## 3.6a Primary sales and free goods
`06` does not define "primary sales (gross, at PTS)". Two readings, both tested: the sum of `net_value_inr`, and invoiced packs x PTS. FOC = free packs x PTS as a share of primary.

In [12]:
ps['q'] = fq(ps.invoice_date)
ps['gross'] = ps.invoiced_qty * ps.pts_inr
ps['foc'] = ps.free_qty * ps.pts_inr
g = ps.groupby('q').agg(net=('net_value_inr', 'sum'), gross=('gross', 'sum'), foc=('foc', 'sum')).loc[QS]
chk = pd.DataFrame({'sum of net_value_inr (cr)': g.net / 1e7, 'invoiced x PTS (cr)': g.gross / 1e7, 'free x PTS / primary %': g.foc / g.gross * 100}).round(2)
chk['QBR primary'] = QBR['Primary sales, gross at PTS (INR crore)']; chk['QBR FOC %'] = QBR['FOC under QS schemes, % of primary']
print('net_value_inr equals invoiced x PTS on every row:', bool(np.allclose(ps.net_value_inr, ps.gross, atol=0.01)))
chk

net_value_inr equals invoiced x PTS on every row: True


,sum of net_value_inr (cr),invoiced x PTS (cr),free x PTS / primary %,QBR primary,QBR FOC %
q,,,,,
FY26Q1,52.52,52.52,4.68,52.52,4.68
FY26Q2,51.84,51.84,3.75,51.84,3.75
FY26Q3,50.42,50.42,3.47,50.42,3.47
FY26Q4,49.37,49.37,3.19,49.37,3.19
FY27Q1,51.36,51.36,3.69,51.36,3.69


## 3.6b Claims logged
The QBR says "NE/expiry claims logged" and does not say how blank values or Rejected claims are treated. Tested readings: (1) `claim_value_inr` as it is; (2) blank values valued at `qty_claimed` x PTS (`01` p3 s5.6: "Values at PTS"); (3) as (2) but leaving out the 31 Rejected claims (all say "Duplicate of earlier claim").

In [13]:
ec['q'] = fq(ec.claim_date)
ec['derived_inr'] = ec.claim_value_derived_inr   # recorded value, or qty x PTS where the register is blank (made in step3_clean)
print('claims with blank value, valued at qty x PTS (claim_value_derived_inr):', int(ec.claim_value_inr.isna().sum()))
print('Rejected claims:', int((ec.status == 'Rejected').sum()), '| remarks:', ec[ec.status == 'Rejected'].remarks.unique().tolist())
nr = ec[ec.status != 'Rejected']
c = pd.DataFrame({
  '(1) as recorded (cr)': ec.groupby('q').claim_value_inr.sum().loc[QS] / 1e7,
  '(2) blanks valued (cr)': ec.groupby('q').derived_inr.sum().loc[QS] / 1e7,
  '(3) blanks valued, no Rejected (cr)': nr.groupby('q').derived_inr.sum().loc[QS] / 1e7}).round(3)
c['QBR (cr)'] = QBR['NE/expiry claims logged (INR crore)']
c['(3) % of primary'] = (nr.groupby('q').derived_inr.sum().loc[QS] / g.gross * 100).round(2)
c['QBR %'] = QBR['Claims as % of primary sales']
c

claims with blank value, valued at qty x PTS (claim_value_derived_inr): 135
Rejected claims: 31 | remarks: ['Duplicate of earlier claim']


,(1) as recorded (cr),(2) blanks valued (cr),"(3) blanks valued, no Rejected (cr)",QBR (cr),(3) % of primary,QBR %
q,,,,,,
FY26Q1,0.610,0.638,0.632,0.62,1.20,1.18
FY26Q2,0.850,0.868,0.859,0.85,1.66,1.64
FY26Q3,1.192,1.236,1.208,1.21,2.40,2.41
FY26Q4,1.117,1.160,1.152,1.15,2.33,2.33
FY27Q1,1.367,1.428,1.414,1.41,2.75,2.75


## 3.6c Fill rate
Short supply is measured per **order line** (`invoice_no` + `sku_code`), because `ordered_qty` sits on the first row only (`01` p3 s5.4). Two readings: invoiced / ordered, and (invoiced + free) / ordered.

In [14]:
ol = ps.groupby(['invoice_no', 'sku_code']).agg(o=('ordered_qty', 'max'), i=('invoiced_qty', 'sum'), f=('free_qty', 'sum'), d=('invoice_date', 'min')).reset_index()
ol['q'] = fq(ol.d)
fr = ol.groupby('q').agg(o=('o', 'sum'), i=('i', 'sum'), f=('f', 'sum')).loc[QS]
fill = pd.DataFrame({'invoiced / ordered %': fr.i / fr.o * 100, '(invoiced + free) / ordered %': (fr.i + fr.f) / fr.o * 100}).round(1)
fill['QBR %'] = QBR['Fill rate to stockists (qty), %']
fill

,invoiced / ordered %,(invoiced + free) / ordered %,QBR %
q,,,
FY26Q1,95.5,99.5,99.5
FY26Q2,91.3,94.6,94.6
FY26Q3,88.6,91.5,91.5
FY26Q4,87.5,90.3,90.3
FY27Q1,88.0,90.8,90.9


## 3.6d Claim-to-credit-note turnaround
Mean days from `claim_date` to `credit_note_date`, for claims that have a credit note. Two ways to put a claim in a quarter: by claim date, or by credit-note date.

In [15]:
cr = ec[ec.credit_note_date.notna()].copy(); cr['days'] = (cr.credit_note_date - cr.claim_date).dt.days; cr['qcn'] = fq(cr.credit_note_date)
tat = pd.DataFrame({'by claim date (mean days)': cr.groupby('q').days.mean().loc[QS], 'by credit-note date (mean days)': cr.groupby('qcn').days.mean().loc[QS]}).round(1)
tat['QBR'] = QBR['Claim-to-CN TAT, days']
tat['policy (days)'] = 30
tat

,by claim date (mean days),by credit-note date (mean days),QBR,policy (days)
FY26Q1,47.1,47.3,47,30
FY26Q2,44.9,45.6,46,30
FY26Q3,44.9,44.0,44,30
FY26Q4,46.7,45.2,45,30
FY27Q1,45.5,45.7,46,30


## 3.6e CFA inventory at quarter end
Quantity on hand x PTS at the quarter-end snapshot. **831 groups have two different quantities for the same date, C&F, SKU and batch** (`qty_conflict`, an open client question). So we compute it four ways and see which one the QBR used.

In [16]:
sn['v'] = sn.qty_on_hand_packs * sn.sku_code.map(PTS)
QEND = {'FY26Q1': '2025-06-30', 'FY26Q2': '2025-09-30', 'FY26Q3': '2025-12-31', 'FY26Q4': '2026-03-31', 'FY27Q1': '2026-06-30'}
k = ['snapshot_date', 'cnf_code', 'sku_code', 'batch_no']
hb = sn[sn.batch_no.notna()]
mx = hb.groupby(k, as_index=False).qty_on_hand_packs.max(); mx['v'] = mx.qty_on_hand_packs * mx.sku_code.map(PTS)
rows = {}
for q, d in QEND.items():
    d = pd.Timestamp(d); s = sn[sn.snapshot_date == d]
    rows[q] = {'all rows added (cr)': s.v.sum() / 1e7, 'conflicting rows left out (cr)': s[~s.qty_conflict].v.sum() / 1e7,
               'larger quantity of each conflict (cr)': mx[mx.snapshot_date == d].v.sum() / 1e7}
inv = pd.DataFrame(rows).T.round(2)
inv['QBR (cr)'] = QBR['CFA inventory, quarter end at PTS (INR crore)']
inv['rows in conflict at that date'] = [int(sn[(sn.snapshot_date == pd.Timestamp(d)) & sn.qty_conflict].shape[0]) for d in QEND.values()]
inv

,all rows added (cr),conflicting rows left out (cr),larger quantity of each conflict (cr),QBR (cr),rows in conflict at that date
FY26Q1,17.49,14.91,17.23,17.49,116
FY26Q2,15.35,12.70,14.97,15.35,116
FY26Q3,13.89,11.33,13.56,13.89,82
FY26Q4,13.14,10.45,12.59,13.14,119
FY27Q1,14.05,10.58,13.32,14.05,148


## 3.6f Claims by therapy area, Q1 FY27 (`06` p2 section 3.1)

In [17]:
QBR_TA = {'Vitamins and nutrition': 46.6, 'Anti-infectives': 39.1, 'Respiratory': 36.6, 'Gastro': 11.2, 'Pain and inflammation': 7.0, 'Cardio-metabolic and CNS': 1.0}
t = nr[nr.q == 'FY27Q1'].merge(sk[['sku_code', 'therapy_area']], on='sku_code')
t['ta'] = t.therapy_area.replace({'Cardio-metabolic': 'Cardio-metabolic and CNS', 'Central nervous system': 'Cardio-metabolic and CNS'})
ta = (t.groupby('ta').derived_inr.sum() / 1e5).round(1).reindex(QBR_TA.keys()).to_frame('ours, reading (3) (INR lakh)')
ta['QBR (INR lakh)'] = list(QBR_TA.values()); ta['difference'] = (ta.iloc[:, 0] - ta['QBR (INR lakh)']).round(1)
ta.loc['Total'] = [ta.iloc[:, 0].sum().round(1), round(sum(QBR_TA.values()), 1), None]
ta

,"ours, reading (3) (INR lakh)",QBR (INR lakh),difference
ta,,,
Vitamins and nutrition,46.2,46.6,-0.4
Anti-infectives,38.8,39.1,-0.3
Respiratory,36.6,36.6,0.0
Gastro,11.2,11.2,0.0
Pain and inflammation,7.1,7.0,0.1
Cardio-metabolic and CNS,1.4,1.0,0.4
Total,141.3,141.5,NaN


## 3.6g Scoreboard: ours against the QBR

In [18]:
def status(ours, qbr, tol, nd):
    d = round(float(ours), nd) - qbr
    return 'MATCH' if abs(d) < 10 ** -nd / 2 else ('CLOSE' if abs(d) <= tol else 'DIFFERS')
SB = []
def add(measure, reading, ours, tol, nd):
    for q, o, b in zip(QS, ours, QBR[measure]):
        SB.append(dict(measure=measure, our_reading=reading, quarter=q, ours=round(float(o), nd), qbr=b, difference=round(float(o) - b, nd + 1), status=status(o, b, tol, nd)))
add('Primary sales, gross at PTS (INR crore)', 'invoiced x PTS = sum of net_value_inr', g.gross / 1e7, 0.005, 2)
add('FOC under QS schemes, % of primary', 'free x PTS / invoiced x PTS', g.foc / g.gross * 100, 0.005, 2)
add('NE/expiry claims logged (INR crore)', '(3) blanks valued at qty x PTS, no Rejected', c['(3) blanks valued, no Rejected (cr)'], 0.015, 2)
add('Claims as % of primary sales', '(3) / primary', c['(3) % of primary'], 0.05, 2)
add('Fill rate to stockists (qty), %', '(invoiced + free) / ordered, per order line', fill['(invoiced + free) / ordered %'], 0.15, 1)
add('Claim-to-CN TAT, days', 'mean days, quarter of credit-note date', tat['by credit-note date (mean days)'], 0.4, 0)
add('CFA inventory, quarter end at PTS (INR crore)', 'qty x PTS, all snapshot rows added', inv['all rows added (cr)'], 0.005, 2)
SB = pd.DataFrame(SB)
print(SB.groupby(['measure', 'status']).size().unstack(fill_value=0).to_string())
SB

status                                         CLOSE  MATCH
measure                                                    
CFA inventory, quarter end at PTS (INR crore)      0      5
Claim-to-CN TAT, days                              0      5
Claims as % of primary sales                       3      2
FOC under QS schemes, % of primary                 0      5
Fill rate to stockists (qty), %                    1      4
NE/expiry claims logged (INR crore)                2      3
Primary sales, gross at PTS (INR crore)            0      5


,measure,our_reading,quarter,ours,qbr,difference,status
0,"Primary sales, gross at PTS (INR crore)",invoiced x PTS = sum of net_value_inr,FY26Q1,52.52,52.52,0.002,MATCH
1,"Primary sales, gross at PTS (INR crore)",invoiced x PTS = sum of net_value_inr,FY26Q2,51.84,51.84,-0.001,MATCH
2,"Primary sales, gross at PTS (INR crore)",invoiced x PTS = sum of net_value_inr,FY26Q3,50.42,50.42,-0.002,MATCH
3,"Primary sales, gross at PTS (INR crore)",invoiced x PTS = sum of net_value_inr,FY26Q4,49.37,49.37,-0.004,MATCH
4,"Primary sales, gross at PTS (INR crore)",invoiced x PTS = sum of net_value_inr,FY27Q1,51.36,51.36,-0.003,MATCH
5,"FOC under QS schemes, % of primary",free x PTS / invoiced x PTS,FY26Q1,4.68,4.68,0.004,MATCH
6,"FOC under QS schemes, % of primary",free x PTS / invoiced x PTS,FY26Q2,3.75,3.75,-0.003,MATCH
7,"FOC under QS schemes, % of primary",free x PTS / invoiced x PTS,FY26Q3,3.47,3.47,-0.004,MATCH
8,"FOC under QS schemes, % of primary",free x PTS / invoiced x PTS,FY26Q4,3.19,3.19,-0.000,MATCH
9,"FOC under QS schemes, % of primary",free x PTS / invoiced x PTS,FY27Q1,3.69,3.69,-0.000,MATCH


## 3.6h What the reconciliation tells us
Read each line against the tables above. The notebook prints the facts; the sentences below are what they support.

In [19]:
f_free = (fill['(invoiced + free) / ordered %'] - fill['QBR %']).abs().max()
f_noFree = (fill['invoiced / ordered %'] - fill['QBR %']).abs().max()
print(f'1. FILL RATE: counting free packs as supplied reproduces the QBR within {f_free:.1f} point; invoiced only is off by up to {f_noFree:.1f} points.')
print('   Free packs were not ordered, so on this reading the QBR fill rate is a little higher than invoiced / ordered (our inference).')
i_all = (inv['all rows added (cr)'] - inv['QBR (cr)']).abs().max(); i_left = (inv['conflicting rows left out (cr)'] - inv['QBR (cr)']).abs().min()
print(f'2. INVENTORY: adding every snapshot row gives the QBR exactly (max gap {i_all:.2f} cr). Leaving the conflicting rows out is at least {i_left:.2f} cr lower.')
print('   So the QBR adds both quantities where one batch has two. Whether that double counts is the open client question.')
print('3. CLAIMS: the QBR is reached only when blank claim values are valued at qty x PTS and the Rejected duplicate claims are left out.')
print('   Recorded values alone understate it. Claims are close, not exact, in FY26 Q1 and Q2.')
print('4. TAT: matches when a claim is counted in the quarter its credit note is dated. It is 14 to 17 days over the 30-day policy in every quarter.')
print('5. PRIMARY and FOC: exact in all 5 quarters, which also confirms that dropping the 746 duplicate invoice rows was right.')

1. FILL RATE: counting free packs as supplied reproduces the QBR within 0.1 point; invoiced only is off by up to 4.0 points.
   Free packs were not ordered, so on this reading the QBR fill rate is a little higher than invoiced / ordered (our inference).
2. INVENTORY: adding every snapshot row gives the QBR exactly (max gap 0.00 cr). Leaving the conflicting rows out is at least 2.56 cr lower.
   So the QBR adds both quantities where one batch has two. Whether that double counts is the open client question.
3. CLAIMS: the QBR is reached only when blank claim values are valued at qty x PTS and the Rejected duplicate claims are left out.
   Recorded values alone understate it. Claims are close, not exact, in FY26 Q1 and Q2.
4. TAT: matches when a claim is counted in the quarter its credit note is dated. It is 14 to 17 days over the 30-day policy in every quarter.
5. PRIMARY and FOC: exact in all 5 quarters, which also confirms that dropping the 746 duplicate invoice rows was right.


## 3.6i Save and validate

In [20]:
SB.to_csv(os.path.join(AUD, 'qbr_reconciliation.csv'), index=False)
sk_ = lambda m: SB[SB.measure == m]
esc = lambda x: str(x).replace('|', '/')
out = ['# QBR reconciliation (Step 3.6)', '',
       'Generated by `notebooks/step3_qbr.ipynb`. QBR figures are from `06_qbr_finance_and_mis_pack.pdf` p1 scorecard; ours are computed from `data-clean/`. Re-run rather than edit.', '',
       'Status: **MATCH** = equal at the QBR rounding; **CLOSE** = within a small tolerance (stated in the notebook); **DIFFERS** = a real gap.', '',
       '| Measure | Our reading | Quarter | Ours | QBR | Diff | Status |', '|---|---|---|---|---|---|---|']
for _, r in SB.iterrows():
    out.append(f'| {esc(r.measure)} | {esc(r.our_reading)} | {r.quarter} | {r.ours:,} | {r.qbr:,} | {r.difference:+} | {r.status} |')
open(os.path.join(NOTES, 'qbr_reconciliation.md'), 'w', encoding='utf-8').write('\n'.join(out))

s = SB.groupby('measure').status.apply(lambda x: x.value_counts().to_dict())
assert (sk_('Primary sales, gross at PTS (INR crore)').status == 'MATCH').all()
assert (sk_('FOC under QS schemes, % of primary').status == 'MATCH').all()
assert (sk_('Claim-to-CN TAT, days').status == 'MATCH').all()
assert (sk_('CFA inventory, quarter end at PTS (INR crore)').status == 'MATCH').all()
assert (sk_('Fill rate to stockists (qty), %').status != 'DIFFERS').all()
assert (sk_('NE/expiry claims logged (INR crore)').status != 'DIFFERS').all()
assert (SB.status != 'DIFFERS').all(), 'a measure differs from the QBR: investigate before moving on'
assert bool(np.allclose(ps.net_value_inr, ps.gross, atol=0.01))
print('ALL RECONCILIATION CHECKS PASSED:', SB.status.value_counts().to_dict())

ALL RECONCILIATION CHECKS PASSED: {'MATCH': 29, 'CLOSE': 6}


**Check:** `ALL RECONCILIATION CHECKS PASSED`. If any line says DIFFERS the assert stops and tells you which measure.